# Build an ORR Activity Volcano


## Goal of the notebook

Let us derive a thermodynamic ORR volcano, place literature metal data on it, and examine uncertainty in the adsorption descriptors.


## Description of the method

For a set of intermediate free energies, the least favorable elementary step sets the limiting potential. Scaling relations express those energies through one OH descriptor. Each branch then follows a different limiting step.

We will apply free energy corrections to BEEF-vdW database values and use its functional ensemble to estimate one part of model uncertainty. The volcano axis shows a calculated limiting potential. It does not represent experimental current or a full kinetic model.


## What we will do

1. Let us derive the limiting potential and draw the volcano from the selected scaling relations.
2. We will load literature data, assemble adsorption free energies, and place metal surfaces on the plot.
3. We will use the functional ensemble to explore descriptor uncertainty and branch changes.


In [ ]:
# Install course dependencies only in Google Colab. Local runs use environment.yml.
!if python -c "import google.colab" >/dev/null 2>&1; then pip install -q ipywidgets ase matplotlib numpy pandas; fi

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ase.db import connect
from functools import partial
from IPython.display import display
from ipywidgets import Checkbox, FloatSlider, HBox, interactive_output, VBox
from pathlib import Path

In [ ]:
# Standard publication figure sizes in inches
plt.rc("figure", figsize  = (8.25 / 2.54, 8.25 / 2.54))
plt.rc("figure", dpi      = 100)
plt.rc("lines",  linewidth= 4)
plt.rc("font",   size     = 10)
plt.rc("font",   family   = "sans-serif")
plt.rc("axes",   titlesize= 10)
plt.rc("axes",   labelsize= 9)
plt.rc("xtick",  labelsize= 8)
plt.rc("ytick",  labelsize= 8)
plt.rc("legend", fontsize = 8)
plt.rc("figure", titlesize= 10)

pd.options.display.float_format = "{:.2f}".format

## Derive the theoretical volcano

**Core exercise**

The scaling relations reduce the ORR free energies to one descriptor:

$$\Delta G_{\mathrm{OOH*}}=\Delta G_{\mathrm{OH*}}+3.2\ \mathrm{eV}$$

$$\Delta G_{\mathrm{O*}}=2\Delta G_{\mathrm{OH*}}.$$

At each descriptor value, the least favorable step sets the limiting potential. The [scaling study by Man, Nørskov, and coauthors](https://doi.org/10.1002/cctc.201000397) provides the basis for this model.

> **Pause and predict:** Which step sets the strong binding branch? Which step sets the weak binding branch?
>
> **What to look for:** Check each branch against the minimum step free energy in the code.


Before we place real catalysts on the diagram, let us draw the theoretical volcano. The scaling relations express each intermediate through $\Delta G_{\mathrm{OH}}$, so this descriptor sets the horizontal position.


In [ ]:
def limiting_potential(delta_g_ooh_ev, delta_g_oh_ev, delta_g_o_ev):
    return min(
        4.92 - delta_g_ooh_ev,
        delta_g_ooh_ev - delta_g_o_ev,
        delta_g_o_ev - delta_g_oh_ev,
        delta_g_oh_ev)

In [ ]:
def plot_theoretical_volcano(beta_ev, selected_oh_ev):
    oh_grid = np.linspace(0, 2, 1000)
    potentials = [limiting_potential(g + beta_ev, g, 2*g) for g in oh_grid]
    selected_v = limiting_potential(selected_oh_ev + beta_ev, selected_oh_ev, 2*selected_oh_ev)
    fig, ax = plt.subplots()
    ax.plot(oh_grid, potentials, color="black")
    ax.scatter(selected_oh_ev, selected_v, color="#1965B0", zorder=3)
    ax.set_xlabel(r"$\Delta G_{\mathrm{OH}}$ / eV")
    ax.set_ylabel("Limiting potential / V")
    ax.set_xlim(0, 2)
    ax.set_ylim(0, 1.5)
    ax.set_title(f"OOH scaling offset = {beta_ev:.2f} eV")
    fig.tight_layout()
    plt.show()
    peak = int(np.argmax(potentials))
    print(f"Sampled peak: {oh_grid[peak]:.2f} eV, {potentials[peak]:.2f} V")

In [ ]:
# Keep the default curve values available for the numerical checkpoint.
oh_energy_grid_ev = np.linspace(0, 2, 1000)
limiting_potential_v = [limiting_potential(g + 3.2, g, 2*g) for g in oh_energy_grid_ev]
volcano_controls = {"beta_ev": FloatSlider(value=3.2, min=2.5, max=3.7, step=0.01, description="β / eV", continuous_update=False), "selected_oh_ev": FloatSlider(value=0.86, min=0.0, max=2.0, step=0.01, description="OH / eV", continuous_update=False),}

In [ ]:
volcano_output = interactive_output(plot_theoretical_volcano, volcano_controls)
display(HBox([volcano_output, VBox(list(volcano_controls.values()))]))

### Find the volcano maximum

Read the optimum from the sampled curve. Its vertical axis gives a thermodynamic limiting potential. The slider reports the peak for the selected $\beta$; the checkpoint below uses $\beta=3.2$ eV.


In [ ]:
# np.argmax returns the index of the largest value in the sampled curve.
peak_index = int(np.argmax(limiting_potential_v))
print("Default β = 3.2 eV: optimum OH descriptor / eV:", round(oh_energy_grid_ev[peak_index], 3))
print("Default β = 3.2 eV: limiting potential / V:", round(limiting_potential_v[peak_index], 3))
print("This ordinate is a potential, not a measured current.")

## Load and correct literature DFT data

**Supporting calculation**

We will download the [BEEF-vdW adsorption-energy database](https://zenodo.org/records/19023993), with energies for metal slabs and adsorbates, plus molecular references for H₂ and H₂O.


In [ ]:
# Download once into the current folder.
!if [ ! -s molecules.db ]; then wget -q 'https://zenodo.org/records/19023993/files/molecules.db?download=1' -O molecules.db; fi
if not Path('molecules.db').is_file() or Path('molecules.db').stat().st_size == 0:
    raise FileNotFoundError("Download failed: molecules.db")
# Download once into the current folder.
!if [ ! -s slab_database.db ]; then wget -q 'https://zenodo.org/records/19023993/files/slab_database.db?download=1' -O slab_database.db; fi
if not Path('slab_database.db').is_file() or Path('slab_database.db').stat().st_size == 0:
    raise FileNotFoundError("Download failed: slab_database.db")
# Download once into the current folder.
!if [ ! -s adsorbate_database.db ]; then wget -q 'https://zenodo.org/records/19023993/files/adsorbate_database.db?download=1' -O adsorbate_database.db; fi
if not Path('adsorbate_database.db').is_file() or Path('adsorbate_database.db').stat().st_size == 0:
    raise FileNotFoundError("Download failed: adsorbate_database.db")

In [ ]:
!ase db adsorbate_database.db --show-values structure_str

Select the slab and adsorbate records we need from the ASE databases. Keep the functional and adsorption site filters visible.


In [ ]:
functional = "BEEF-vdW"
molecule_rows = []
for row in connect("molecules.db").select(xc=functional):
    molecule_rows.append(row)
slab_rows = []
for row in connect("slab_database.db").select(xc=functional):
    slab_rows.append(row)
adsorbate_rows = []
for row in connect("adsorbate_database.db").select(xc=functional):
    adsorbate_rows.append(row)

Let us retrieve the molecular reference energies used in each adsorption equation.


In [ ]:
for row in molecule_rows:
    if row.formula == "H2":
        energy_h2_ev = row.energy
    elif row.formula == "H2O":
        energy_h2o_ev = row.energy

Create dictionaries keyed by structure name. Then we can match slab, adsorbate, and reference energies by name.


In [ ]:
slab_energies_ev = {}
for row in slab_rows:
    slab_energies_ev[row.key_value_pairs["structure_str"]] = row.get("energy")
adsorbate_energies_ev = {}
for row in adsorbate_rows:
    adsorbate_energies_ev[row.key_value_pairs["structure_str"]] = row.get("energy")

In [ ]:
o_fcc_names = []
for name in adsorbate_energies_ev:
    if "_O_fcc" in name and adsorbate_energies_ev[name] is not None:
        o_fcc_names.append(name)
oh_top_names = []
for name in adsorbate_energies_ev:
    if "_OH_top" in name and adsorbate_energies_ev[name] is not None:
        oh_top_names.append(name)
ooh_top_names = []
for name in adsorbate_energies_ev:
    if "_OOH_top" in name and adsorbate_energies_ev[name] is not None:
        ooh_top_names.append(name)
oh_top_names.remove("Ni_111_OH_top")
ooh_top_names.remove("Ni_111_OOH_top")

Free energy corrections come from this [source for ORR free energy corrections](https://doi.org/10.1021/acssuschemeng.8b04173).

Here, SC denotes the solvent correction.

| Intermediate | SC / eV | $\Delta ZPE - T\Delta S$ / eV |
|:---|:---:|:---:|
| *O | 0.00 | 0.05 |
| *OH | −0.50 | 0.35 |
| *OOH | −0.30 | 0.40 |


Now we have the raw DFT energies. Let us add the literature ZPE, entropy, and solvation terms to obtain adsorption free energies for O, OH, and OOH.


In [ ]:
# Reference O*, OH*, and OOH* to H₂O and H₂ within the CHE convention.
o_free_energies_ev = {}
for name in o_fcc_names:
    slab = name.replace("_O_fcc", "")
    electronic_adsorption_energy_ev = (adsorbate_energies_ev[name] - slab_energies_ev[slab] - (energy_h2o_ev - energy_h2_ev))
    o_free_energies_ev[name] = electronic_adsorption_energy_ev + 0.05
oh_free_energies_ev = {}
for name in oh_top_names:
    slab = name.replace("_OH_top", "")
    electronic_adsorption_energy_ev = (adsorbate_energies_ev[name] - slab_energies_ev[slab] - (energy_h2o_ev - 0.5 * energy_h2_ev))
    oh_free_energies_ev[name] = electronic_adsorption_energy_ev + 0.35 - 0.5
ooh_free_energies_ev = {}

In [ ]:
for name in ooh_top_names:
    slab = name.replace("_OOH_top", "")
    electronic_adsorption_energy_ev = (adsorbate_energies_ev[name] - slab_energies_ev[slab] - (2 * energy_h2o_ev - 1.5 * energy_h2_ev))
    ooh_free_energies_ev[name] = electronic_adsorption_energy_ev + 0.4 - 0.3
rows = []
for name in oh_free_energies_ev:
    o_name = name.replace("OH_top", "O_fcc")
    ooh_name = name.replace("OH", "OOH")
    rows.append({"name": name, "metal": name.split("_111")[0], "dG_O": o_free_energies_ev[o_name], "dG_OH": oh_free_energies_ev[name], "dG_OOH": ooh_free_energies_ev[ooh_name], "dG_O_sd": np.nan, "dG_OH_sd": np.nan, "dG_OOH_sd": np.nan,})
volcano_results = pd.DataFrame(rows)

We will collect each metal's adsorption free energies in a table. The color shows its offset from the scaling relation.


In [ ]:
# .copy() creates an independent object so changes do not overwrite the original.
table = volcano_results.copy()
table["dG_O - 2dG_OH / eV"] = table["dG_O"] - 2 * table["dG_OH"]
table["dG_OOH - dG_OH - 3.2 / eV"] = table["dG_OOH"] - table["dG_OH"] - 3.2
columns = ["metal", "dG_OH", "dG_O", "dG_OOH", "dG_O - 2dG_OH / eV", "dG_OOH - dG_OH - 3.2 / eV",]
display(table[columns].sort_values("dG_OH", ascending=False).round(2))

With the adsorption free energies ready, let us place each metal on the theoretical volcano and inspect its calculated limiting potential.


In [ ]:
# .iterrows() visits a table one row at a time; row contains the named column values.
def plot_volcano(results, beta_ev=3.2):
    colors = {"Pt": "#1d6996", "Cu": "#EF553B", "Pd": "#332288", "Rh": "#24796c", "Ag": "#b3b3b3", "Ir": "#666666", "Au": "#e6ab02",}
    fig, ax = plt.subplots()
    ax.set_xlabel("dG_OH / eV")
    ax.set_ylabel("The limiting potential / V")
    ax.set_ylim(bottom=0)
    oh_energy_grid_ev = np.linspace(0, 2, 1000)
    limiting_potential_v = []
    for oh_energy_ev in oh_energy_grid_ev:
        limiting_potential_v.append(limiting_potential(oh_energy_ev + beta_ev, oh_energy_ev, oh_energy_ev * 2))
    ax.plot(oh_energy_grid_ev, limiting_potential_v, c="black")
    for _, row in results.iterrows():
        metal = row["metal"]
        limiting_potential_v = limiting_potential(row["dG_OOH"], row["dG_OH"], row["dG_O"])
        ax.scatter(row["dG_OH"], limiting_potential_v, label=metal, c=colors[metal])
        if (not np.isnan(row["dG_OH_sd"]) and (not np.isnan(row["dG_OOH_sd"])) and (not np.isnan(row["dG_O_sd"]))):
            # Pair each step energy with its uncertainty, then select the limiting step.
            step_energies_and_uncertainties = [
                (4.92 - row["dG_OOH"], row["dG_OOH_sd"]),
                (row["dG_OOH"] - row["dG_O"], (row["dG_OOH_sd"]**2 + row["dG_O_sd"]**2)**0.5),
                (row["dG_O"] - row["dG_OH"], (row["dG_O_sd"]**2 + row["dG_OH_sd"]**2)**0.5),
                (row["dG_OH"], row["dG_OH_sd"]),
            ]
            limiting_step = step_energies_and_uncertainties[0]
            for step in step_energies_and_uncertainties[1:]:
                if step[0] < limiting_step[0]:
                    limiting_step = step
            yerr = limiting_step[1]
            ax.errorbar(row["dG_OH"], limiting_potential_v, xerr=row["dG_OH_sd"], yerr=yerr, ecolor=colors[metal])
    ax.legend()
    fig.show()

### Explore selected metal surfaces

**Core exercise**

Change $\beta$ to move the theoretical curve. The source energies for each catalyst stay fixed.


In [ ]:
# Source catalyst energies stay fixed; β changes only the theoretical reference curve.
comparison_controls_1 = {"beta_ev": FloatSlider(value=3.2, min=2.5, max=3.7, step=0.01, description="β / eV", continuous_update=False)}
comparison_plot_1 = partial(plot_volcano, results=volcano_results.copy())
comparison_output_1 = interactive_output(comparison_plot_1, comparison_controls_1)
display(HBox([comparison_output_1, VBox(list(comparison_controls_1.values()))]))

## Estimate ensemble uncertainty

The BEEF-vdW ensemble samples one part of exchange correlation model uncertainty. We will use its spread to estimate uncertainty for the volcano points. On the strong binding branch, the limiting potential follows $\Delta G_{\mathrm{OH}}$, so its uncertainty moves along that branch.

> **Pause and predict:** Which catalysts may change branch across their uncertainty range?
>
> **What to look for:** Treat local error propagation with care near the volcano peak and branch boundaries.


BEEF-vdW provides 2000 related exchange correlation parameterizations. Their energies give us a sample of one model uncertainty source. The database keeps matching reference, slab, and adsorbate entries in a shared order, so we can combine corresponding values and estimate a spread for each catalyst.


In [ ]:
mol_ens = {}
for row in molecule_rows:
    mol_ens[row.formula] = row.data["ensemble_en"]
met_ens = {}
for row in slab_rows:
    if "ensemble_en" in dict(row.data):
        met_ens[row.key_value_pairs["structure_str"]] = row.data["ensemble_en"]
ads_ens = {}
for row in adsorbate_rows:
    if "ensemble_en" in dict(row.data):
        ads_ens[row.key_value_pairs["structure_str"]] = row.data["ensemble_en"]

In [ ]:
# .loc[condition, column] selects rows by a Boolean condition and named columns.
oh_ensemble_free_energies_ev = {}
oh_uncertainties_ev = {}
for name in oh_top_names:
    slab = name.replace("_OH_top", "")
    if name in ads_ens and slab in met_ens:
        ensemble_adsorption_energies_ev = (ads_ens[name] - met_ens[slab] - (mol_ens["H2O"] - 0.5 * mol_ens["H2"]))
        oh_uncertainties_ev[name] = ensemble_adsorption_energies_ev.std()
        oh_ensemble_free_energies_ev[name] = (ensemble_adsorption_energies_ev + 0.35 - 0.5)
        volcano_results.loc[volcano_results["name"] == name, "dG_OH_sd"] = (oh_uncertainties_ev[name])

In [ ]:
ooh_ensemble_free_energies_ev = {}
ooh_uncertainties_ev = {}
for name in ooh_top_names:
    slab = name.replace("_OOH_top", "")
    if name in ads_ens and slab in met_ens:
        ensemble_adsorption_energies_ev = (ads_ens[name] - met_ens[slab] - (2 * mol_ens["H2O"] - 1.5 * mol_ens["H2"]))
        ooh_uncertainties_ev[name] = ensemble_adsorption_energies_ev.std()
        ooh_ensemble_free_energies_ev[name] = (ensemble_adsorption_energies_ev + 0.4 - 0.3)
        volcano_results.loc[volcano_results["name"].str.startswith(slab), "dG_OOH_sd"] = ooh_uncertainties_ev[name]

In [ ]:
o_ensemble_free_energies_ev = {}
o_uncertainties_ev = {}
for name in o_fcc_names:
    slab = name.replace("_O_fcc", "")
    if name in ads_ens and slab in met_ens:
        ensemble_adsorption_energies_ev = (ads_ens[name] - met_ens[slab] - (mol_ens["H2O"] - mol_ens["H2"]))
        o_uncertainties_ev[name] = ensemble_adsorption_energies_ev.std()
        o_ensemble_free_energies_ev[name] = ensemble_adsorption_energies_ev + 0.05
        volcano_results.loc[volcano_results["name"].str.startswith(slab), "dG_O_sd"] = (o_uncertainties_ev[name])

We now have uncertainty estimates for O, OH, and OOH. Let us update the table and draw the volcano points with error bars. In this dataset, the O values have a broader ensemble spread than the OH and OOH values.


In [ ]:
table = volcano_results.copy()
table["dG_O - 2dG_OH / eV"] = table["dG_O"] - 2 * table["dG_OH"]
table["dG_OOH - dG_OH - 3.2 / eV"] = table["dG_OOH"] - table["dG_OH"] - 3.2
columns = ["metal", "dG_OH", "dG_O", "dG_OOH", "dG_O_sd", "dG_OH_sd", "dG_OOH_sd", "dG_O - 2dG_OH / eV", "dG_OOH - dG_OH - 3.2 / eV",]
display(table[columns].sort_values("dG_OH", ascending=False).round(2))

At a branch crossing, `min()` switches from one limiting step to another. Local error propagation assumes one branch applies across the uncertainty range. This assumption is least secure near the peak and branch boundaries. On the strong binding branch, the limiting potential depends on $\Delta G_{\mathrm{OH}}$, so the uncertainty follows the branch.


In [ ]:
# Source catalyst energies stay fixed; β changes only the theoretical reference curve.
comparison_controls_2 = {"beta_ev": FloatSlider(value=3.2, min=2.5, max=3.7, step=0.01, description="β / eV", continuous_update=False)}
comparison_plot_2 = partial(plot_volcano, results=volcano_results.copy())
comparison_output_2 = interactive_output(comparison_plot_2, comparison_controls_2)
display(HBox([comparison_output_2, VBox(list(comparison_controls_2.values()))]))

## Student exercise: Move the volcano optimum

**Core exercise**

Change the OOH scaling offset from 3.2 to 3.0 eV in the theoretical expression. Find the peak for each value and identify the limiting step on both sides. Explain why a change in limiting potential alone cannot tell us the size of an experimental current.

Record a prediction, one numerical or graphical result, and your chemical interpretation. Keep the stored values intact.


In [ ]:
# Your code here.
# Use a separate variable for your changed parameter and display the comparison.

### Your answer

Prediction: [write your prediction]

Result: [report a value or refer to your figure]

Chemical explanation: [explain the result]

Assumption or limitation: [state one limitation]


## What have you learned

Let us gather the main ideas from this notebook. You can now:

- Explain which step sets each branch of a thermodynamic volcano.
- Calculate limiting potentials from corrected intermediate energies.
- Read deviations of metal surfaces from an ideal scaling curve.
- Distinguish functional ensemble spread from experimental variation and explain the limits of local error propagation near the apex.


### Questions for discussion

1. Which step sets each branch? How does a change in the OOH scaling offset move the optimum?
2. Why can the limiting potential uncertainty become asymmetric near a branch crossing?
